# Demo 6: Tool integration (MCP on Databricks)

In Session 1 the agent's tools were plain functions, and MCP was a protocol between a local client and a local server. Here we give the same Python wiki agent its tools **on Databricks**, one route at a time. This notebook follows the reference doc [**github.com/pranav-platformatory/agent-framework-in-databricks/docs/tool-integration-in-databricks.md**](https://github.com/pranav-platformatory/agent-framework-in-databricks/blob/main/docs/tool-integration-in-databricks.md):

| Part | Topic | Doc section |
|---|---|---|
| T0 | **Tool inventory**: which tool takes which route | Tools for agents on Databricks |
| T1 | **Tool creation, without UC and with UC** | UC functions as tools |
| T2 onward | Custom MCP server, MCP from the agent, governing tools, end to end | *(later notebooks / sections)* |

**The agent changes in one way.** Before, `answer_question` always called `search_wiki` first, then made one LLM call. Now the **model chooses** its tools in a loop (as in Session 1), because discovery and choice are what MCP is about.

**Before running**: `common/governance_config.py` must define `CATALOG`, `SCHEMA` and `GATEWAY_MODEL` (the model service from Demo 5). Its model must support **tool calling**; T0 checks this. `common/wiki_search.py` (`search_wiki` as a plain function, JSON-safe, no MLflow) and `common/exercises.py` (one practice exercise per wiki page) are new. T1 needs **serverless compute** for the UC function calls. A SQL warehouse alone is not enough [1].

[n] refers to **References** at the end.

In [0]:
%pip install -U "mlflow[databricks]>=3.1" openai "unitycatalog-ai[databricks]" "unitycatalog-openai[databricks]"
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.0 MB/s  0:00:00
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.122.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-9b5d0bcf-c27e-4c97-83cf-431362ac8518
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 146.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 114.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 107.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 137.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 103.5 MB/s  0:00:00
   ━━━━━

In [0]:
%pip install -U databricks-sdk
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
  Using cached http://package-proxy.host.local:8184/pypi/v1/packages/databricks-sdk/0.156.0/databricks_sdk-0.156.0-py3-none-any.whl.metadata (44 kB)
Using cached http://package-proxy.host.local:8184/pypi/v1/packages/databricks-sdk/0.156.0/databricks_sdk-0.156.0-py3-none-any.whl (1.1 MB)
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.94.0
    Uninstalling databricks-sdk-0.94.0:
      Successfully uninstalled databricks-sdk-0.94.0
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import mlflow
import openai
import pandas as pd
from databricks.sdk import WorkspaceClient

from common import wiki_search
from common.config import EXPERIMENT_PATH
from common.exercises import EXERCISES, exercise_text
from common.governance_config import CATALOG, GATEWAY_MODEL, SCHEMA
from common.wiki_pages import WIKI_PAGES

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT_PATH)
mlflow.openai.autolog()  # every LLM call becomes a child span
w = WorkspaceClient()

# The LLM is reached through a model service (Demo 5), not a serving endpoint.
# Same token and client setup as Demo 5.
os.environ["DATABRICKS_TOKEN"] = os.environ.get("DATABRICKS_TOKEN") or w.config.authenticate()["Authorization"].removeprefix("Bearer ")
gw = openai.OpenAI(api_key=os.environ["DATABRICKS_TOKEN"], base_url=f"{w.config.host.rstrip('/')}/ai-gateway/mlflow/v1")
TOOL_MODEL = GATEWAY_MODEL  # any model service whose model supports tool calling

assert set(EXERCISES) == set(WIKI_PAGES), "common/exercises.py needs one exercise per wiki page id"


def fn_spec(name, description, properties, required):
    """A tool, as the model sees it: a name, a description, and an input schema."""
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {"type": "object", "properties": properties, "required": required},
        },
    }


print("Tool model:", TOOL_MODEL)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


Tool model: main.new_python_wiki_agent.wiki_agent_llm


# T0. Tool inventory

A **tool** is a name, a description the model reads to decide when to use it, and an input schema. That is exactly what MCP's `tools/list` returns in Session 1. On Databricks there are several routes to the same thing; they differ in *where the tool lives* and *what governs it* [3][4]. The tutor's tools, and the route each will take across this demo:

| Tool | What it does | Route | Part |
|---|---|---|---|
| `search_wiki(query)` | Explain a concept: find wiki pages by keyword | Local Python function, then a **custom MCP server on Databricks Apps** | T1, then T2 |
| `get_exercise(topic_id)` | Give a practice exercise and a hint for a topic | Local Python function, then a **UC SQL function** | T1 |
| `run_python(code)` | Run a learner's code in isolation | **Databricks-provided MCP** `system.ai.sandbox` (Unity Gateway beta) | T3, T6 |

**Each tool answers a kind of question the others cannot.** That is what makes the model's choice visible in a trace: a tool is only called when a question needs it.

Session 1 mapped onto Databricks:

| Session 1 (localhost) | On Databricks |
|---|---|
| Host: our agent loop | Our agent loop: a notebook now, a served agent later |
| Client: `client.py` | `DatabricksMCPClient`, or a framework's MCP integration (T3) |
| Server: `servers.py` on `127.0.0.1` | A Databricks-provided MCP, a registered external one, or a custom server on Databricks Apps |
| STDIO / HTTP transport | Streamable HTTP only |
| Anyone on the machine can call it | UC privileges, OAuth, Unity Gateway controls |

## T0.1 Which MCP services does this workspace have?

Databricks-provided MCPs are named `system.ai.<name>`. Which ones you can use depends on region, entitlements and previews. The UI page is **Unity Gateway > MCPs** [3]. The CLI equivalent is `databricks ai-gateway list-mcp-services --parent schemas/system.ai` [5].

In [0]:
import databricks.sdk

print("databricks-sdk", databricks.sdk.version.__version__)
if hasattr(w, "ai_gateway"):
    for s in w.ai_gateway.list_mcp_services(parent="schemas/system.ai"):
        print("-", s.name, "|", s.comment or "")
else:  # older SDKs have no `ai_gateway` API (it exists in 0.156.0)
    print("This databricks-sdk has no `ai_gateway` API. Upgrade it, restart Python, and re-run the setup cell:")
    print("  %pip install -U databricks-sdk   then   dbutils.library.restartPython()")
    print("Or use Unity Gateway > MCPs in the UI, or in a terminal:")
    print("  databricks ai-gateway list-mcp-services --parent schemas/system.ai")

databricks-sdk 0.156.0
- mcp-services/system.ai.google_drive | Connects to the third-party Google Drive, Docs, Sheets, and Slides APIs. Verify that the provider's service and your intended use meet your organization's compliance requirements. [Documentation](https://docs.databricks.com/aws/en/agents/mcp-tools/built-in-mcp-services#external-services)
- mcp-services/system.ai.github | Connects to the third-party GitHub MCP server. Verify that the provider's service and your intended use meet your organization's compliance requirements. [Documentation](https://docs.databricks.com/aws/en/agents/mcp-tools/built-in-mcp-services#external-services)
- mcp-services/system.ai.web_search | Search the public web.
- mcp-services/system.ai.atlassian | Connects to Jira and Confluence through the third-party Atlassian Rovo MCP server. Verify that the provider's service and your intended use meet your organization's compliance requirements. [Documentation](https://docs.databricks.com/aws/en/agents/mcp-t

## T0.2 Check that the model can call tools

Everything from here on needs a model that supports tool calling. One cheap probe now saves a confusing failure later.

In [0]:
probe = gw.chat.completions.create(
    model=TOOL_MODEL,
    messages=[{"role": "user", "content": "What is 2 + 3? Use the add tool."}],
    tools=[fn_spec("add", "Add two numbers.", {"a": {"type": "number"}, "b": {"type": "number"}}, ["a", "b"])],
    temperature=0,
)
assert probe.choices[0].message.tool_calls, (
    f"{TOOL_MODEL} did not call the tool. Point GATEWAY_MODEL at a model service whose model supports tool calling."
)
print("Tool call requested:", probe.choices[0].message.tool_calls[0].function)

Tool call requested: Function(arguments='{"a": 2, "b": 3}', name='add')


Trace(trace_id='tr-8d668007e4f64f3aab227c20b960d4a3')

# T1. Tool creation: without UC, then with UC

## T1.1 The agent loop

The loop from Session 1, on Databricks. The model sees the tool list; if it asks for a tool, we run it and send the result back; if not, that is the answer. Two things are traced:

- `call_tool`: The **one place every tool is called**
  > So every tool call is a `TOOL` span, whichever route the tool takes.
- `run_agent`: The `AGENT` root span.
  > The LLM calls come from autolog.

A tool is a dict `{"spec": ..., "run": ...}`: what the model sees, and what we run.

***That one shape is what lets later parts mix routes in one loop.***

> **NOTE**: The prompt does not name the tools. The model has to choose from their descriptions alone, which is the point of T1.6.

In [0]:
TUTOR_PROMPT = """You are a Python tutor for beginners. Use your tools when they help.
If the wiki does not cover a question, say so. Do not add links or sources that are not in the wiki."""

MAX_STEPS = 6


def call_tool(tools, name, args):
    """Run one tool. Every tool call goes through here, so each one is a TOOL span."""
    with mlflow.start_span(name=name, span_type="TOOL") as span:
        span.set_inputs(args)
        result = tools[name]["run"](**args)
        span.set_outputs(result)
        return result


@mlflow.trace(span_type="AGENT")
def run_agent(question, tools, verbose=True):
    specs = [t["spec"] for t in tools.values()]
    messages = [{"role": "system", "content": TUTOR_PROMPT}, {"role": "user", "content": question}]
    calls = []
    for _ in range(MAX_STEPS):
        resp = gw.chat.completions.create(model=TOOL_MODEL, messages=messages, tools=specs, temperature=0.2)
        msg = resp.choices[0].message
        if not msg.tool_calls:  # no tool requested: this is the answer
            return {"response": msg.content, "tool_calls": calls}
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            name, args = tc.function.name, json.loads(tc.function.arguments or "{}")
            try:
                result = call_tool(tools, name, args)
            except Exception as e:  # the TOOL span is already marked as an error; tell the model too
                result = f"Error: {e}"
            calls.append({"tool": name, "args": args})
            if verbose:
                print(f"  -> {name}({args})\n  <- {str(result)[:150]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result if isinstance(result, str) else json.dumps(result),
            })
    return {"response": "(stopped: too many tool calls)", "tool_calls": calls}


def try_questions(tools, questions):
    """Ask each question and show which tools the model chose to use."""
    rows = []
    for q in questions:
        out = run_agent(q, tools, verbose=False)
        rows.append({
            "question": q,
            "tools used": ", ".join(c["tool"] for c in out["tool_calls"]) or "(none)",
            "answer (start)": out["response"][:140].replace("\n", " "),
        })
    display(pd.DataFrame(rows))


## T1.2 Without UC: tools as plain Python functions

The Session 1 way. Each tool is a function in our code, and we write the description the model reads by hand. Two tools, for two kinds of question:

- `search_wiki`: explanations, from the wiki
- `get_exercise`: practice exercises, which the wiki does **not** hold

In [0]:
LOCAL_TOOLS = {
    "search_wiki": {
        "spec": fn_spec(
            "search_wiki",
            "Search the Python wiki by keywords. Returns the best-matching pages as {id, page_content, score}.",
            {"query": {"type": "string", "description": "Keywords for the topic, e.g. 'add item list'"}},
            ["query"],
        ),
        "run": lambda query: wiki_search.search_wiki(query),
    },
    "get_exercise": {
        "spec": fn_spec(
            "get_exercise",
            "Get one practice exercise and a hint for a Python topic. Use it when the learner asks for an exercise or practice.",
            {"topic_id": {"type": "string", "description": f"Topic id, one of: {', '.join(EXERCISES)}"}},
            ["topic_id"],
        ),
        "run": lambda topic_id: exercise_text(topic_id),
    },
}

QUESTIONS = [
    "How do I add an item to a list?",
    "Give me a practice exercise on dicts",
    "How do I handle errors with try/except?",
    "What is a generic in Java?",
]
try_questions(LOCAL_TOOLS, QUESTIONS)

question,tools used,answer (start)
How do I add an item to a list?,(none),"To add an item to a list in Python, you can use the `append()` method. Here's a simple example: ```python my_list = [1, 2, 3] my_list.appen"
Give me a practice exercise on dicts,get_exercise,"Here's a practice exercise on dictionaries: ### Exercise Make a dictionary of three people and their ages. Print each as `name: age`, then"
How do I handle errors with try/except?,get_exercise,"To handle errors with try/except in Python, you can use the following structure: ```python try: # Code that might raise an exception exc"
What is a generic in Java?,search_wiki,"The wiki does not cover Java generics. However, I can tell you that Java generics allow for the creation of reusable code. They enable types"


[Trace(trace_id='tr-7902f7a4dd9a8e95eda5c09c688ef72e'), Trace(trace_id='tr-abd946aa290432d2d3abe703087d44ba'), Trace(trace_id='tr-7f77807299618565fe90bb4ab49eddc3'), Trace(trace_id='tr-6f2a3abc17ac7eb592da94c05d1f56c9')]

**Look at**: the table above, then **Experiments** and the latest traces. What we expected, and what to compare it with:

| Question | Kind | Tool we expect |
|---|---|---|
| How do I add an item to a list? | Covered by the wiki | `search_wiki` |
| Give me a practice exercise on dicts | Needs data the wiki does not hold | `get_exercise` |
| How do I handle errors with try/except? | Deliberately **not** in the wiki | `search_wiki`, which finds nothing useful |
| What is a generic in Java? | Out of scope | Probably none |

The model's choice is its own: the table shows what it **did**, not what we hoped. Each trace has an `AGENT` root span, `LLM` spans from autolog, and a `TOOL` span per call, named after the tool. Open one and read its inputs and outputs. What is missing? Nothing here is *governed*. The tools are lines of code in this notebook: whoever runs the notebook can run them, and another agent that wants `get_exercise` has to copy the code.

## T1.3 With UC: the same tool as a function in Unity Catalog

A UC function is a securable (`catalog.schema.function`) that an agent can call as a tool. It suits predefined logic where the model only supplies the arguments [1][2]. We move `get_exercise` there:

1. Put the exercises in a UC **table** (the function needs something to query)
2. Create a SQL **function** over it.
    > **NOTE**: The `COMMENT`s are the descriptions the model will read: the function's says what it does, each parameter's says what to pass [2].

In [0]:
EXERCISES_TABLE = f"{CATALOG}.{SCHEMA}.exercises"
FN_EXERCISE = f"{CATALOG}.{SCHEMA}.get_exercise"

rows = [(topic, e["exercise"], e["hint"]) for topic, e in EXERCISES.items()]
spark.createDataFrame(rows, "topic string, exercise string, hint string").write.mode("overwrite").saveAsTable(EXERCISES_TABLE)
display(spark.table(EXERCISES_TABLE))

topic,exercise,hint
variables,"Store your name and age in two variables, then print them on one line.","Assign with `=`, then pass both variables to `print()`."
data_types,"Convert the string '42' to an int, add 8, and print the result and its type.",Use `int()` and `type()`.
strings,"Given `name = 'ada'`, print `Hello, ADA!` using an f-string and `.upper()`.",An f-string can call a method inside the braces.
lists,"Start with `nums = [3, 1, 2]`. Add 4, remove 1, then print the list and its length.","Use `append`, `remove` and `len`."
dictionaries,"Make a dictionary of three people and their ages. Print each as `name: age`, then look up a person who is not in it without raising an error.",Loop with `.items()`; look up with `.get()` and a default.
conditionals,"Print `positive`, `zero` or `negative` for a number `x`.","Use `if`, `elif` and `else`."
loops,"Print the numbers 0 to 4. Then loop over `[1, 2, 3, 4]` and stop as soon as you reach 3.",Use `range(5)` for the first part and `break` for the second.
functions,Write `greet(name='friend')` that returns a greeting. Call it with and without an argument.,"A function without `return` gives back `None`, so remember to `return`."
modules,Use the `random` module to print a random whole number from 1 to 6.,"Start with `import random`, then look for `randint`."
files,"Write two lines to `notes.txt`, then read the file back and print it.","Use `with open(...)` in mode `'w'`, then open the file again to read it."


In [0]:
TOPIC_IDS = ", ".join(EXERCISES)

# The parameter is `topic_id`, not `topic`: a parameter named like a column would shadow it in the query.
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FN_EXERCISE}(
  topic_id STRING COMMENT 'Topic id, one of: {TOPIC_IDS}. These are the same ids as the wiki pages. Use the closest id if the learner words it differently.'
)
RETURNS STRING
COMMENT 'Returns one practice exercise and a hint for a Python topic. Use it when the learner asks for an exercise or practice.'
RETURN SELECT COALESCE(MAX(concat('Exercise: ', exercise, ' Hint: ', hint)), 'No exercise for that topic.')
FROM {EXERCISES_TABLE} WHERE topic = topic_id
""")

# Test it with a plain SELECT before handing it to an agent [2].
display(spark.sql(f"SELECT {FN_EXERCISE}('loops') AS exercise"))

exercise
"Exercise: Print the numbers 0 to 4. Then loop over `[1, 2, 3, 4]` and stop as soon as you reach 3. Hint: Use `range(5)` for the first part and `break` for the second."


**Look at**: **Catalog Explorer**, `catalog > schema > Functions > get_exercise`. The `COMMENT`s are the metadata. They are also exactly what the model will be given as the tool description: the next cell shows that.

## T1.4 Give the UC function to the agent

`UCFunctionToolkit` reads the function from UC and builds the tool spec. It does **not** run the function: `DatabricksFunctionClient` does that, on serverless compute [1]. The toolkit does its lookup **once, when it is built**: the list is whatever `function_names` resolves to at that moment (a name like `catalog.schema.*` expands to every function in the schema). **A running agent does not discover functions**: adding one to the schema later does not add it to an agent that already has its tools.

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from unitycatalog.ai.openai.toolkit import UCFunctionToolkit

uc_client = DatabricksFunctionClient()


def uc_runner(function_name):
    def run(**args):
        result = uc_client.execute_function(function_name=function_name, parameters=args)
        if result.error:
            raise RuntimeError(result.error)
        return result.value

    return run


def uc_tools(function_names):
    """UC functions as agent tools: {tool name: {"spec", "run"}}, the same shape as LOCAL_TOOLS."""
    toolkit = UCFunctionToolkit(function_names=function_names, client=uc_client)
    tools = {}
    for fn, spec in toolkit.tools_dict.items():  # keyed by the full UC function name
        name = spec["function"]["name"]
        assert name.endswith(fn.split(".")[-1]), f"Tool {name} does not match function {fn}; check the toolkit's naming"
        tools[name] = {"spec": spec, "run": uc_runner(fn)}
    return tools


exercise_tool = uc_tools([FN_EXERCISE])
for t in exercise_tool.values():
    print(json.dumps(t["spec"], indent=2))

/local_disk0/.ephemeral_nfs/envs/pythonEnv-9b5d0bcf-c27e-4c97-83cf-431362ac8518/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


{
  "type": "function",
  "function": {
    "name": "main__new_python_wiki_agent__get_exercise",
    "strict": true,
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "topic_id": {
          "anyOf": [
            {
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "description": "Topic id, one of: variables, data_types, strings, lists, dictionaries, conditionals, loops, functions, modules, files. These are the same ids as the wiki pages. Use the closest id if the learner words it differently.",
          "title": "Topic Id"
        }
      },
      "title": "main__new_python_wiki_agent__get_exercise__params",
      "type": "object",
      "required": [
        "topic_id"
      ]
    },
    "description": "Returns one practice exercise and a hint for a Python topic. Use it when the learner asks for an exercise or practice."
  }
}


Compare this with the hand-written spec in T1.2: the description and the parameter description are the `COMMENT`s from the function. Now ask the exercise question again, with `search_wiki` still local and `get_exercise` from UC. This is the first time the UC function really runs.

In [0]:
tools_uc = {"search_wiki": LOCAL_TOOLS["search_wiki"], **exercise_tool}
out = run_agent("Give me a practice exercise on dicts", tools_uc)
print("\nAnswer:", out["response"])

  -> main__new_python_wiki_agent__get_exercise({'topic_id': 'dictionaries'})
  <- Exercise: Make a dictionary of three people and their ages. Print each as `name: age`, then look up a person who is not in it without raising an error

Answer: Here's a practice exercise on dictionaries:

Make a dictionary of three people and their ages. Print each as `name: age`, then look up a person who is not in it without raising an error.

**Hint:** Loop with `.items()`; look up with `.get()` and a default.

You can try to solve this exercise on your own, and I'll be happy to help if you need any guidance or want to check your solution.


Trace(trace_id='tr-a7f560329cbc59adf6a4c5ee4eeaf602')

**Look at**: the new trace. The second tool now runs as a UC function on serverless compute, and the `TOOL` span carries the UC-derived name (`catalog__schema__get_exercise`). Check the arguments: the learner said `dicts`, and the `COMMENT` told the model which topic ids exist.

What changed, and what did not:

| | Local Python function (T1.2) | UC function (T1.3-T1.4) |
|---|---|---|
| Where it lives | In the agent's code | UC: `catalog.schema.function` |
| What the model reads | A hand-written spec | The function's and parameters' `COMMENT`s |
| Who can run it | Whoever runs the code | Whoever holds `EXECUTE` (next cell) |
| Reuse | Copy the code | Any agent can reference the name |
| Runs on | The agent's process | Serverless general compute [1] |
| Tool discovery | A list fixed in code | A list fixed when the toolkit is built (`function_names`) |

The last row is the gap that MCP fills (T3).

## T1.5 Grant and revoke `EXECUTE` (changes real permissions)

To call a UC function you need `EXECUTE` on it, plus `USE CATALOG` and `USE SCHEMA` on its parents [1]. As in Demo 5, set `PRINCIPAL` to a principal you intend to grant access to.

> **Note**: The owner of a function can always run it. If `PRINCIPAL` is you, this cell shows the grants changing but no call fails. To see a call fail, use a second user or a service principal.

In [0]:
PRINCIPAL = "pranav@platformatory.com"  # TODO
assert not PRINCIPAL.startswith("<"), "Set PRINCIPAL first"

spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{PRINCIPAL}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA} TO `{PRINCIPAL}`")
spark.sql(f"GRANT EXECUTE ON FUNCTION {FN_EXERCISE} TO `{PRINCIPAL}`")
display(spark.sql(f"SHOW GRANTS ON FUNCTION {FN_EXERCISE}"))

In [0]:
# Take it back. `REVOKE` succeeds even if the privilege was not granted, so it is safe to re-run.
spark.sql(f"REVOKE EXECUTE ON FUNCTION {FN_EXERCISE} FROM `{PRINCIPAL}`")
display(spark.sql(f"SHOW GRANTS ON FUNCTION {FN_EXERCISE}"))

## T1.6 Break it: a vague `COMMENT`

The model never sees our code or our table. The `COMMENT`s are **all it knows** about the tool. Here is the same function with careless descriptions, under a vague name, asked the same question:

In [0]:
FN_VAGUE = f"{CATALOG}.{SCHEMA}.lookup"

spark.sql(f"""
CREATE OR REPLACE FUNCTION {FN_VAGUE}(x STRING COMMENT 'input')
RETURNS STRING
COMMENT 'Looks something up.'
RETURN SELECT COALESCE(MAX(concat('Exercise: ', exercise, ' Hint: ', hint)), 'Nothing found.')
FROM {EXERCISES_TABLE} WHERE topic = x
""")

vague_tool = uc_tools([FN_VAGUE])
out = run_agent("Give me a practice exercise on dicts", {"search_wiki": LOCAL_TOOLS["search_wiki"], **vague_tool})
print("\nAnswer:", out["response"])


Answer: ## Exercise: Working with Dictionaries

### Task

Create a dictionary to store information about a person. The dictionary should have the following keys: `name`, `age`, `city`. 

1. Initialize the dictionary with some sample data.
2. Print out the person's name and age.
3. Update the person's age to 30.
4. Add a new key-value pair to the dictionary: `country` = `USA`.
5. Print out the entire dictionary.

### Your Turn

Try to complete the task on your own before checking the solution.

### Solution

```python
# Initialize the dictionary
person = {
    'name': 'John Doe',
    'age': 25,
    'city': 'New York'
}

# Print out the person's name and age
print(person['name'])
print(person['age'])

# Update the person's age to 30
person['age'] = 30

# Add a new key-value pair to the dictionary
person['country'] = 'USA'

# Print out the entire dictionary
print(person)
```

Let's check if this code works as expected. We can run it using the ipython tool.

[ipython(code="person = {'name

Trace(trace_id='tr-0152db81319bd57054a4e25cb937530a')

**Look at**: which tools were called, and with what arguments. Compare with T1.4, where the answer came from the function. With nothing saying what `x` is or which values exist, the model typically does one of two things:

- It passes the learner's wording (`'dicts'`) and gets *Nothing found.*
- It never calls the tool, and **invents an exercise from its own memory**: no tool call in the trace, and an answer the function never produced.

Either way the function was fine and the answer was worse, because of the `COMMENT`s. The result varies by model and run, so rehearse this one and keep a trace of a bad run.

The lesson: for a UC function tool, **the `COMMENT`s are the prompt.** Write them for a reader who can see nothing else, and list the valid values.

## What T1 left open

- Both routes use a **tool list fixed when the agent is built**. A tool added later is invisible until we rebuild the agent.
- Neither route puts the tool **behind a server** that many agents can discover and call.

MCP addresses both. T2 turns `search_wiki` into an MCP server on Databricks Apps; T3 lets the agent discover its tools from it.

In [0]:
# `lookup` was a prop for T1.6. `get_exercise` and the `exercises` table stay: later parts reuse them.
spark.sql(f"DROP FUNCTION IF EXISTS {FN_VAGUE}")

# Left over from the first version of this notebook (a wiki table and function that nothing needs any more):
spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.get_wiki_page")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.wiki_pages")

# References

Docs checked 2026-10-24. Unity Gateway features marked Beta can change.

1. [Create agent tools using Unity Catalog functions](https://docs.databricks.com/aws/en/agents/custom-agents/create-custom-tool): privileges, serverless compute, `UCFunctionToolkit`
2. [Connect agents to structured data](https://docs.databricks.com/aws/en/agents/custom-agents/structured-retrieval-tools): SQL functions as tools, `COMMENT`s as descriptions, test with `SELECT`
3. [MCP on Databricks](https://docs.databricks.com/aws/en/agents/mcp-tools/): the kinds of MCP server, **Unity Gateway > MCPs**
4. [Tool integration in Databricks](https://github.com/pranav-platformatory/agent-framework-in-databricks/blob/main/docs/tool-integration-in-databricks.md): the reference doc for this demo
5. [Use MCP tools in a Python agent](https://docs.databricks.com/aws/en/agents/custom-agents/use-mcp-tools): `databricks ai-gateway list-mcp-services`